In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
# @title
!pip install --upgrade pip
!pip install --upgrade jiwer evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 19.1 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 55.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [evaluate]


In [3]:
import datasets
import transformers
import soundfile as sf
import librosa
import re
import pandas as pd
from torch.utils.tensorboard import SummaryWriter
import jiwer
import evaluate
import torch
import os



In [4]:
from huggingface_hub import notebook_login
notebook_login()

In [1]:
import datasets
import evaluate
import re
import gc
import os

# Disable HF default cache so we can manually control disk usage
datasets.disable_caching()

NUM_MAP = {
    "1": "motsi", "2": "piri", "3": "tatu", "4": "ina", "5": "shanu",
    "6": "tanhatu", "7": "nomwe", "8": "tsere", "9": "pfumbamwe", "0": "zero"
}

def clean_text_pipeline(text):
    if text is None: return ""
    text = text.lower()
    for num, word in NUM_MAP.items():
        text = text.replace(num, f" {word} ")
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def create_dataset_ultra_safe(dataset_id, language, sample_rate, max_audio_len=30.0, min_audio_len=5.0):
    print(f"Loading '{language}' dataset...")

    # 1. Load splits directly
    train_pattern = f"data/ASR/{language}/{language}-train-*.parquet"
    val_pattern = f"data/ASR/{language}/{language}-validation-*.parquet"

    ds_train = datasets.load_dataset(dataset_id, name=f"{language}_asr", data_files={"train": train_pattern}, split="train")
    ds_val = datasets.load_dataset(dataset_id, name=f"{language}_asr", data_files={"validation": val_pattern}, split="validation")

    ds_combined = datasets.concatenate_datasets([ds_train, ds_val])

    # 2. Resample
    print(f"Resampling audio to {sample_rate}Hz...")
    ds_combined = ds_combined.cast_column("audio", datasets.Audio(sampling_rate=sample_rate))

    # 3. Filter Lengths - We save the output locally to clear RAM
    print("Filtering audio lengths...")
    def filter_audio_length(example):
        arr = example["audio"]["array"]
        duration = len(arr) / sample_rate
        return min_audio_len <= duration <= max_audio_len

    ds_filtered = ds_combined.filter(filter_audio_length)

    # Clear RAM of the massive combined dataset
    del ds_combined
    gc.collect()

    # 4. Clean Text
    print("Cleaning text...")
    def clean_batch_text(batch):
        batch["transcription"] = [clean_text_pipeline(t) for t in batch["transcription"]]
        return batch

    ds_cleaned = ds_filtered.map(clean_batch_text, batched=True)

    # Clear RAM of the filtered dataset
    del ds_filtered
    gc.collect()

    # 5. Split Dataset
    print("Performing standard 80/20 train/test split...")
    final_splits = ds_cleaned.train_test_split(test_size=0.2, seed=42)

    return final_splits

# Run dataset loading
lang_code = "sna"
sr = 16000
max_len = 30.0

# If we already have the split on disk from a previous run, load it!
if os.path.exists("./waxal_processed_dataset"):
    print("Found processed dataset on disk! Loading...")
    prepared_dataset = datasets.load_from_disk("./waxal_processed_dataset")
else:
    print("No processed dataset found. Running safe pipeline...")
    prepared_dataset = create_dataset_ultra_safe("google/WaxalNLP", lang_code, sr, max_len)
    print("Saving processed dataset to disk to protect against future crashes...")
    prepared_dataset.save_to_disk("./waxal_processed_dataset")

wer_metric = evaluate.load("wer")

# Force final garbage collection
gc.collect()

print("✅ Dataset preparation finished safely!")
print(f"Train samples: {len(prepared_dataset['train'])}, Val samples: {len(prepared_dataset['test'])}")

No processed dataset found. Running safe pipeline...
Loading 'sna' dataset...
Resampling audio to 16000Hz...
Filtering audio lengths...


Filter:   0%|          | 0/15836 [00:00<?, ? examples/s]

Cleaning text...


Map:   0%|          | 0/14958 [00:00<?, ? examples/s]

Performing standard 80/20 train/test split...
Saving processed dataset to disk to protect against future crashes...


Saving the dataset (0/8 shards):   0%|          | 0/11966 [00:00<?, ? examples/s]

Saving the dataset (0/2 shards):   0%|          | 0/2992 [00:00<?, ? examples/s]

✅ Dataset preparation finished safely!
Train samples: 11966, Val samples: 2992


In [6]:
import torch
import datasets
import evaluate
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)
from transformers.trainer_utils import IntervalStrategy # Added import
from dataclasses import dataclass
from typing import Any, Dict, List, Union

# 1. Load the clean dataset directly from your Colab disk (Super fast!)
print("Loading saved dataset from disk...")
dataset = datasets.load_from_disk("./waxal_processed_dataset")

# 2. Load Processor and Model
model_id = "openai/whisper-small"
print(f"Loading {model_id}...")
processor = WhisperProcessor.from_pretrained(model_id, language="Shona", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(model_id)

# Force model to not generate token IDs that shouldn't be predicted
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

# 3. Create the Custom Data Collator (The Memory Saver!)
# This processes audio into spectrograms ONLY when the batch is passed to the GPU.
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Extract raw audio arrays and text from the batch
        input_features = [{"input_features": self.processor.feature_extractor(feature["audio"]["array"], sampling_rate=feature["audio"]["sampling_rate"]).input_features[0]} for feature in features]
        label_features = [{"input_ids": self.processor.tokenizer(feature["transcription"]).input_ids} for feature in features]

        # Pad the audio features to the maximum length expected by the model (30s)
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # Pad the text labels to the maximum length in the batch
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # Replace padding with -100 so the loss function ignores it
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # If bos token is appended in previous tokenization step, cut it off
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

# 4. Define Training Arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-small-waxal",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=1e-5,
    warmup_steps=20,
    max_steps=100,
    gradient_checkpointing=True,
    fp16=True,
    eval_strategy="steps",
    per_device_eval_batch_size=2,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=50,
    eval_steps=50,
    logging_steps=10,
    report_to=["tensorboard"],
    save_total_limit=1,
    remove_unused_columns=False,
)

# 5. Initialize Trainer
trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
)

print("Starting Memory-Safe Whisper Training...")
trainer.train()


Loading saved dataset from disk...
Loading openai/whisper-small...


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Starting Memory-Safe Whisper Training...


Step,Training Loss,Validation Loss
50,8.801228,1.038302
100,7.400237,0.901669


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=100, training_loss=10.96960075378418, metrics={'train_runtime': 970.192, 'train_samples_per_second': 1.649, 'train_steps_per_second': 0.103, 'total_flos': 4.61736640512e+17, 'train_loss': 10.96960075378418, 'epoch': 0.13371218452281464})